In [1]:
import json
from monty.json import MontyDecoder
import plotly.graph_objects as go
import plotly.express as px
import os
from pymatgen.core import Element


from utils_kga.statistical_analysis.get_spin_and_bond_angle_statistics import *
from utils_kga.general import pretty_plot

In [2]:
# Load edge-df
# replace to change between analyzing either cryst. uniques or all structures
# with open("data/dfs_of_magnetic_edge_information.json") as f:
with open("data/dfs_of_magnetic_edge_information_include_crystallographic_multiples.json") as f: 
    dict_all_stats = json.load(f)
all_stats = {key: pd.DataFrame.from_dict(df) for key, df in dict_all_stats.items()}

# For metadata filtering and computation of occurrences in magnetic primitive cells
with open("../../data_retrieval_and_preprocessing_MAGNDATA/data/df_grouped_and_chosen_commensurate_MAGNDATA.json") as f:
    df = json.load(f, cls=MontyDecoder)

plot_dir = "plots/TM_octahedra_eg-non-eg"
os.makedirs(plot_dir, exist_ok=True)

In [3]:
# Add is_tm bool for later easier analysis
for ang_df in all_stats.values():
    ang_df["site_is_tm"] = ang_df["site_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["site_to_is_tm"] = ang_df["site_to_element"].apply(lambda el: Element(el).is_transition_metal)
    ang_df["ligand_el_set"] = ang_df["ligand_elements"].apply(lambda ls: set(ls))

    ang_df["site_ion"] = ang_df.apply(lambda r: r["site_element"] + str(int(r["site_oxidation"])) + "+", axis=1)
    ang_df["site_to_ion"] = ang_df.apply(lambda r: r["site_to_element"] + str(int(r["site_to_oxidation"])) + "+", axis=1)

In [ ]:
# d1, d2, d3 listed by Goodenough in "Magnetism and the Chemical Bond" as illustrative cations
# plus "Ru5+", "Os5+" as appeared in prev. analysis of ion pairs at larger bond angles
non_eg_ions = ["Ti3+", "V4+", "Nb4+", "Re6+", 
               "Ti2+", "V3+", "Cr4+", "Ru6+", "Os6+", 
               "V2+", "Cr3+", "Mn4+", "Mo3+", "W3+", "Re4+", "Ir6+",
               "Ru5+", "Os5+"] 
 
# hs-d5, hs-d6, d7, d8, d9 listed by Goodenough in "Magnetism and the Chemical Bond" as illustrative cations
eg_ions = ["Mn2+", "Fe3+", 
           "Fe2+", "Co3+", 
           "Co2+", "Ni3+", 
           "Ni2+", "Pd2+", "Pt2+", "Au3+",
           "Cu2+"]


In [5]:
ligand_multiplicity_bool = False
ligand_multiplicity_string = "no ligand multiplicity included"
normalize_bool = False
normalize_string = "absolute occurrences"
data_string = "all edges with selected eg-non-eg TM octahedra at both nodes (incl. cryst multiples)"

entries = 0
almost_collinear_entries = 0
all_spin_occus = []
for md_id, ang_df in all_stats.items():
    subdf = ang_df.loc[(ang_df["site_is_tm"])
            & (ang_df["site_to_is_tm"])
    ]
    subdf = subdf.loc[(subdf["site_ce"]=="O:6") & (subdf["site_to_ce"]=="O:6")]
    subdf = subdf.loc[
        ((subdf["site_ion"].isin(eg_ions)) & (subdf["site_to_ion"].isin(non_eg_ions))) | ((subdf["site_to_ion"].isin(eg_ions)) & (subdf["site_ion"].isin(non_eg_ions)))]

    if not subdf.empty:
        print(md_id, set(subdf["site_ion"].values))
        entries += 1
        n_lattice_points = df.at[md_id, "n_lattice_points"]
        occus = get_bond_angle_occurrences(df=subdf,
                                                            include_ligand_multiplicity=ligand_multiplicity_bool,
                                                            normalize=normalize_bool,
                                                            n_lattice_points=n_lattice_points,
                                                            spin_angle_round=0,
                                                            bond_angle_round=7)
        all_spin_occus.extend(occus)
        if any([True for e in occus if (e[0] >= 170.0 or e[0] <= 10.0)]):
            almost_collinear_entries += 1
all_spin_occus_df = pd.DataFrame(columns=["spin_angle", "bond_angle", "occurrence"], data=all_spin_occus)

for spin_ang_lim in [10.0, 170.0]:
    if spin_ang_lim == 10.0:
        sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] <= spin_ang_lim]
    else:
        sub_df = all_spin_occus_df.loc[all_spin_occus_df["spin_angle"] >= spin_ang_lim]

    spin_ang_lim_string = f">=_{spin_ang_lim}_spin_angle" if spin_ang_lim == 170.0 else f"<=_{spin_ang_lim}_spin_angle"

    one_d_fig = go.Figure(layout=go.Layout(xaxis=go.layout.XAxis(title="Bond angle (°)"),
                                            yaxis=go.layout.YAxis(title="Occurrence")))

    one_d_fig.add_trace(go.Histogram(
        histfunc="sum",
        x=sub_df["bond_angle"].values,
        y=sub_df["occurrence"].values,
        nbinsx=181,
        name=spin_ang_lim,
        marker_color="#025268"
    ))
    one_d_fig = pretty_plot(one_d_fig)
    one_d_fig.update_layout(title=dict(
        text=f"{data_string} ({entries} structures ({almost_collinear_entries} almost coll.), {np.sum(sub_df.occurrence.values)} / {np.sum(all_spin_occus_df.occurrence.values)} occurrences), {ligand_multiplicity_string}, {normalize_string}, spin_ang_lim = {spin_ang_lim}°",
    font=dict(size=12)))
    one_d_fig.update_layout(xaxis_range=[125, 182], yaxis_range=[0, 42])
    one_d_fig.update_yaxes(zeroline=False)
    # one_d_fig.show()
    one_d_fig.write_image(os.path.join(plot_dir, f"bond_angle_distributions_{spin_ang_lim_string}_"
                                       f"eg-non-eg_TM_oct_{normalize_string}_{ligand_multiplicity_string}.pdf"))


0.164 {'Co2+', 'Mn4+'}
0.204 {'Re6+', 'Mn2+'}
0.205 {'Re6+', 'Mn2+'}
0.210 {'Os6+', 'Co2+'}
0.551 {'Re6+', 'Mn2+'}
0.572 {'Cr3+', 'Ni2+'}
0.573 {'Cr3+', 'Ni2+'}
0.682 {'Os5+', 'Fe3+'}
0.748 {'Ru5+', 'Ni2+'}
0.749 {'Ru5+', 'Co2+'}
0.750 {'Ru5+', 'Co2+'}
0.796 {'Os6+', 'Ni2+'}
1.32 {'Co2+', 'Mn4+'}
1.46 {'Os5+', 'Fe3+'}
1.47 {'Os5+', 'Fe3+'}
1.565 {'Os6+', 'Co2+'}
2.18 {'Mn4+', 'Ni2+'}
